<a id="top"></a>
# Paper 3 — `Ortho-Unified: Orthogonal Query-Routed Canonical + CORN Metric Bridge over GeoFiLM`

Notebook ini menggunakan **struktur, split, prompt, target prosa murni, HP, dan inisialisasi RNG yang 100% identik** dengan `01_GeoFiLM_Paper2_Master_Ablation.ipynb` (`M4_GeoFiLM`) dan `05_GeoFiLM_CanonBridge.ipynb` (`B1_M4_CanonBridge`), lalu mengganti jembatan kanonik dengan **`OrthoUnifiedBridge`** (`GeoFiLM_Paper2/ortho_unified_core.py`) untuk melampaui `Gautam_Qwen2.5VL_7B` ($85,50\%$ Qwen / $87,02\%$ Llama) sebesar **$>3\%$ (target $88,5\%\text{--}90,0\%$)**.

### Rantai Bukti Empiris & 5 Perbaikan Matematis `Ortho-Unified`
Pada eksperimen Paper 2+, `B1_M4_CanonBridge` sudah mengalahkan `Gautam_Qwen2.5VL_7B` ($85,96\%$ vs $85,50\%$ Qwen-Judge, $p=0,011$; $87,50\%$ vs $87,02\%$ Llama-Judge, $p=0,0053$), namun diagnostik `[E4]` notebook 05 mengungkap **5 leher botol** yang menahan kenaikannya di $+0,48\%$:

| # | Leher Botol pada `B1_M4_CanonBridge` (Bukti `[E4]` Notebook 05) | Solusi Matematis pada `Ortho-Unified` (`ortho_unified_core.py`) |
|---|---|---|
| **1** | **Kepala lokasi buta koordinat 2D:** `CanonicalBridge` hanya memberi 4 angka area (`mean, std, max, ent`) tanpa $(\mu_x, \mu_y)$. Akibatnya 10 kepala non-lokasi mencapai **90,8%–99,8%**, tetapi 3 kepala lokasi tertinggal di **55,2%–69,3%**. | **9D Spatio-Geometric Moments + 2D Fourier Coordinates per Category:** Setiap kategori menerima $[\mu_{x,c}, \mu_{y,c}, \sigma_{xx,c}, \sigma_{yy,c}, \sigma_{xy,c}, \mu_g, \sigma_g, s_{\max}, \mathcal{H}] \in \mathbb{R}^9$ + fitur Fourier 2D dari $(\mu_{x,c}, \mu_{y,c})$ ($16\text{D}$) ke `fc[ci]` dan `embed[ci]`. |
| **2** | **Penjumlahan buta 16 kategori (derau 15:1):** Semua 16 kategori dijumlahkan ke `tok0` tanpa melihat pertanyaan $Q$, sehingga gerbang tertahan kecil ($|g|=0,073 / 0,055$). | **Token-Level Query Cross-Attention & Supervised Query-Aspect Router $r_c(Q) \in (0,1)$:** Memprediksi aspek yang ditanyakan $Q$ (disupervisi BCE terhadap `canon_m`) dengan bobot routing residual $w_c(Q) = 0,25 + 1,75\,r_c(Q)$. |
| **3** | **Tabrakan fitur Bridge vs GeoFiLM:** Vektor bridge ditambahkan langsung di atas `e_total` dan `pe_2d`, memicu gerbang spasial negatif (`-0,11` / `-0,13`). | **Proyeksi Subruang Ortogonal Gram-Schmidt Eksak (Pulau FP32):** Memproyeksikan $\mathbf{m}_{\text{glob}}^\perp \perp \mathbf{e}_{\text{total}}$ dan $\mathbf{m}_{\text{spat},p}^\perp \perp \mathbf{pe}_{\text{2d},p}$ sehingga $\langle \mathbf{m}^\perp, \mathbf{e} \rangle = 0$ eksak di setiap step. |
| **4** | **`polyp_size` diperlakukan tak-berurutan (BCE):** Tertinggal $9,5\text{ pp}$ dibanding `E_CLBRIDGE_r32` ($72,7\%$ vs $82,2\%$). | **Dedicated Monotonic CORN + Squared EMD Ordinal Head:** Menjamin probabilitas kumulatif monotonik $P(Y > k) = \prod_{j\le k}\sigma(z_j)$ pada `<5mm < 5-10mm < 11-20mm < >20mm` dan memadukannya ke kepala `polyp_size`. |
| **5** | **Modulasi spasial seragam antar patch:** Injeksi spasial hanya bergantung pada skalar $a_c[p]$. | **Per-Patch Coordinate-Aware Residual FiLM:** Modulasi $(\gamma_p, \beta_p)$ berbasis koordinat 2D patch $+ \mathbf{m}_{\text{spat},p}^\perp$ berinisialisasi nol (Step 0 $\equiv$ M4 persis, `val_L_VQA = 5.0737`). |

* **Folder output:** `experiments/Paper3_Ortho_Unified_r32`
* **Modul inti:** `GeoFiLM_Paper2/ortho_unified_core.py` (`CORE_VERSION = "ortho-unified-2026-11-15"`)
* **Evaluasi terpisah:** `GeoFiLM_Paper2/07_Ortho_Unified_Evaluation.ipynb`

### Urutan Menjalankan
| Mode | Cell |
|---|---|
| **FRESH** (run baru) | `[A1]` → restart runtime → `[A2]`–`[A11]` → `[B1]` → `[D1]` → `[D2]` |
| **RESUME** (terputus) | `[A2]`–`[A10]` → `[C1]` → `[D1]` → `[D2]` |
| **INFERENSI TEST** (setelah training) | `[A2]`–`[A9]` → `[E1]` → `[E2]` → lanjut ke `07_Ortho_Unified_Evaluation.ipynb` |

# PART A — COMMON SETUP, LABEL KANONIK + ORDINAL & ARSITEKTUR ORTHO-UNIFIED
<a id="part-a"></a>

# [A1] Environment — install paket

In [1]:
# [A1] Environment — install paket & restart runtime sekali (identik notebook 01 & 05)
!pip install -q transformers==4.49.0 peft "torchao>=0.16.0" rouge-score sacrebleu nltk scikit-learn scipy
import transformers, peft
print("transformers:", transformers.__version__, "| peft:", peft.__version__)
print("\n>>> JIKA BARU INSTAL PAKET, RESTART RUNTIME SEKARANG, lalu lanjutkan mulai dari Cell [A2] <<<")

transformers: 4.49.0 | peft: 0.21.2

>>> JIKA BARU INSTAL PAKET, RESTART RUNTIME SEKARANG, lalu lanjutkan mulai dari Cell [A2] <<<


# [A2] Import & cek versi pustaka

In [1]:
# [A2] Import & cek versi pustaka
import transformers, peft, torch, gc
print("transformers:", transformers.__version__, "| peft:", peft.__version__, "| torch:", torch.__version__)
assert transformers.__version__ == "4.49.0", "Versi transformers harus 4.49.0 (sama dengan run M4 & B1)."

transformers: 4.49.0 | peft: 0.21.2 | torch: 2.14.1+cu130


# [A3] Mount Google Drive

In [2]:
# [A3] Mount Google Drive (otomatis dilewati jika berjalan di mesin lokal Windows/Linux)
import os
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
print("Running on Colab:", IS_COLAB)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running on Colab: True


# [A4] Konfigurasi path, seed, & caching gambar

In [3]:
# [A4] Konfigurasi path, seed, & caching gambar ke disk lokal (identik notebook 01 & 05, OUTDIR baru)
from pathlib import Path
import json, re, math, random, os, shutil, time
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

if IS_COLAB:
    BASE = Path("/content/drive/MyDrive/DOKTORAL/KvasirVQAX1")
    LOCAL = Path("/content/all_images")
else:
    BASE = Path(r"g:/My Drive/DOKTORAL/KvasirVQAX1")
    LOCAL = BASE / "all_images"

SRC_DIR  = BASE / "all_images"
ZIP_PATH = BASE / "all_images.zip"

t0_img = time.time()
if IS_COLAB and (not LOCAL.exists() or len(os.listdir(LOCAL)) < 6449):
    LOCAL.mkdir(parents=True, exist_ok=True)
    if ZIP_PATH.exists():
        print(f"Menyalin arsip {ZIP_PATH.name} dari Google Drive ke disk lokal VM...")
        shutil.copy2(ZIP_PATH, "/content/all_images.zip")
        print("Mengekstrak gambar ke disk lokal (/content/all_images)...")
        os.system("unzip -q -o /content/all_images.zip -d /content/all_images")
        if os.path.exists("/content/all_images.zip"):
            os.remove("/content/all_images.zip")
        print(f"✓ Selesai ekstrak dalam {time.time() - t0_img:.1f} detik! Total gambar: {len(os.listdir(LOCAL))}")
    else:
        from tqdm.auto import tqdm
        src_files = [f for f in os.listdir(SRC_DIR) if not f.startswith(".")]
        loc_files = set(os.listdir(LOCAL))
        missing_files = [f for f in src_files if f not in loc_files]
        for f in tqdm(missing_files, desc="Salin Gambar", unit="file"):
            shutil.copy2(SRC_DIR / f, LOCAL / f)
        print(f"✓ Selesai salin dalam {time.time() - t0_img:.1f} detik! Total gambar: {len(os.listdir(LOCAL))}")
else:
    print(f"✓ Folder gambar siap: {LOCAL} ({len(os.listdir(LOCAL)) if LOCAL.exists() else 0} file).")

IMG_DIR    = LOCAL
A0_R32_DIR = BASE / "experiments/Stage2_CORAL_linear/prediction_rank32/20260916-060029_VQAonly-r32_a0.0_s42"
SPLIT_PATH = BASE / "experiments/Stage2_CORAL_linear/val_split_images.json"
BASE_ID    = "microsoft/Florence-2-base-ft"
CORE_DIR   = BASE / "GeoFiLM_Paper2"                       # lokasi ortho_unified_core.py
M4_DIR     = BASE / "experiments/Paper2_GeoFiLM_r32"       # run M4 (pembanding LIVE)
B1_DIR     = BASE / "experiments/Paper2plus_Bridge_r32"    # run B1_M4_CanonBridge (pembanding LIVE)
OUTDIR     = BASE / "experiments/Paper3_Ortho_Unified_r32"
OUTDIR.mkdir(parents=True, exist_ok=True)

if SPLIT_PATH.exists() and not (OUTDIR / "val_split_images.json").exists():
    shutil.copy2(SPLIT_PATH, OUTDIR / "val_split_images.json")

for name, p in [("all_images", IMG_DIR), ("A0_vqaonly_r32", A0_R32_DIR), ("split", SPLIT_PATH),
                ("core_dir", CORE_DIR), ("m4_dir", M4_DIR), ("b1_dir", B1_DIR)]:
    print("OK  " if p.exists() else "MISS", f"{p}")
print("OK   OUTDIR ->", OUTDIR)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", dev, torch.cuda.get_device_name(0) if dev == "cuda" else "")

✓ Folder gambar siap: /content/all_images (6449 file).
OK   /content/all_images
OK   /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Stage2_CORAL_linear/prediction_rank32/20260916-060029_VQAonly-r32_a0.0_s42
OK   /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Stage2_CORAL_linear/val_split_images.json
OK   /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/GeoFiLM_Paper2
OK   /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper2_GeoFiLM_r32
OK   /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper2plus_Bridge_r32
OK   OUTDIR -> /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper3_Ortho_Unified_r32
device: cuda NVIDIA A100-SXM4-80GB


# [A5] Warisi Hyperparameter & konfigurasi LoRA r=32

In [4]:
# [A5] Warisi Hyperparameter & konfigurasi LoRA r=32 langsung dari Baseline A0 (identik notebook 01 & 05)
A0_CFG_PATH  = A0_R32_DIR / "config.json"
A0_LORA_PATH = A0_R32_DIR / "checkpoints/final/adapter_config.json"

if A0_CFG_PATH.exists():
    a0_meta = json.loads(A0_CFG_PATH.read_text())
    HP = a0_meta["hp"]
    print(f"Loaded HP from A0 VQA-only r=32 ({A0_R32_DIR.name}):")
else:
    HP = {
        "lr": 1e-05, "batch_size": 16, "grad_accum": 2, "epochs": 5.0,
        "warmup_ratio": 0.1, "weight_decay": 0.01, "scheduler": "linear",
        "fp16": True, "bf16": False
    }
    print("Loaded canonical A0 VQA-only r=32 HP:")

for k, v in HP.items():
    print(f"  {k:14s} {v}")

if A0_LORA_PATH.exists():
    lora_cfg = json.loads(A0_LORA_PATH.read_text())
else:
    lora_cfg = {
        "r": 32, "lora_alpha": 64, "lora_dropout": 0.05,
        "target_modules": ["q_proj", "qkv", "proj", "v_proj"],
        "bias": "none", "task_type": "SEQ_2_SEQ_LM"
    }

print("\nLoRA r=32 config (from A0 VQA-only r=32):")
for k in ["r", "lora_alpha", "lora_dropout", "target_modules", "bias", "task_type"]:
    print(f"  {k:16s} {lora_cfg.get(k)}")
assert int(lora_cfg["r"]) == 32 and int(lora_cfg["lora_alpha"]) == 64, f"Expected r=32, got {lora_cfg}"

Loaded HP from A0 VQA-only r=32 (20260916-060029_VQAonly-r32_a0.0_s42):
  lr             1e-05
  batch_size     16
  grad_accum     2
  epochs         5.0
  warmup_ratio   0.1
  weight_decay   0.01
  scheduler      linear
  fp16           True
  bf16           False

LoRA r=32 config (from A0 VQA-only r=32):
  r                32
  lora_alpha       64
  lora_dropout     0.05
  target_modules   ['qkv', 'q_proj', 'proj', 'v_proj']
  bias             none
  task_type        SEQ_2_SEQ_LM


# [A6] Muat dataset train & terapkan Image-Disjoint Validation Split

In [5]:
# [A6] Muat dataset train & terapkan Image-Disjoint Validation Split yang 100% identik dengan M4 & B1
from datasets import load_from_disk

ds_tr = load_from_disk(str(BASE / "train"))
tr = ds_tr.to_pandas()
print("Total train rows:", len(tr))

strip_jpg = lambda s: re.sub(r"\.(jpg|jpeg|png)$", "", str(s), flags=re.I)
tr["_img"] = tr["img_id"].map(strip_jpg)

assert SPLIT_PATH.exists(), f"Split file tidak ditemukan di {SPLIT_PATH}"
val_imgs = set(json.loads(SPLIT_PATH.read_text())["val_images"])
tr["_split"] = np.where(tr["_img"].isin(val_imgs), "val", "train")

overlap = set(tr.loc[tr._split == "train", "_img"]) & set(tr.loc[tr._split == "val", "_img"])
assert len(overlap) == 0, "Image overlap antara train dan val harus 0!"

for s in ["train", "val"]:
    g = tr[tr._split == s]
    print(f"  {s:5s} rows={len(g):7d}  unique_images={g._img.nunique():5d}")
assert (tr._split == "train").sum() == 129023 and (tr._split == "val").sum() == 14571, \
    "Ukuran split berbeda dari run M4/B1 — perbandingan tidak lagi apple-to-apple."

Total train rows: 143594
  train rows= 129023  unique_images= 5590
  val   rows=  14571  unique_images=  622


# [A7] Muat `ortho_unified_core.py`, gerbang versi, & label kanonik + ordinal

In [6]:
# [A7] Muat ortho_unified_core.py, GERBANG VERSI, & bangun label kanonik + CORN ordinal dari split TRAIN
import sys, importlib

EXPECTED_CORE_VERSION = "ortho-unified-2026-11-15"
assert (CORE_DIR / "ortho_unified_core.py").exists(), \
    f"ortho_unified_core.py tidak ada di {CORE_DIR}. Pastikan file modul ada di folder GeoFiLM_Paper2."
if str(CORE_DIR) not in sys.path:
    sys.path.insert(0, str(CORE_DIR))
import ortho_unified_core as OUC
importlib.reload(OUC)
assert OUC.CORE_VERSION == EXPECTED_CORE_VERSION, (
    f"Versi modul di Drive = {OUC.CORE_VERSION!r}, seharusnya {EXPECTED_CORE_VERSION!r}.\n"
    "Drive belum tersinkron: tunggu 1-2 menit, Runtime > Restart, lalu ulangi dari [A2].")
OUC.selfcheck_as_list()
from ortho_unified_core import (
    GeoFiLMModule, OrthoUnifiedBridge, GeoFiLMOrthoUnified, CanonLabeler,
    canonical_loss, ortho_unified_loss, canonical_predict, canonical_batch_stats,
    discover_canonical_specs, as_list, atoms, as_class_list, aspect_pairs,
    POLYP_SIZE_ORDERED_TIERS
)
print("✓ ortho_unified_core", OUC.CORE_VERSION, "| as_list lulus uji format asli dataset\n")

CANON_MIN_COUNT, CANON_MIN_COVER = 20, 0.99
CANON_SPECS = discover_canonical_specs(tr[tr._split == "train"],
                                       min_count=CANON_MIN_COUNT, min_cover=CANON_MIN_COVER)
LABELER = CanonLabeler(CANON_SPECS)

tr["_canon"] = [LABELER.encode(q, o) for q, o in zip(tr["question_class"], tr["original"])]

asked, got = {c: 0 for c in LABELER.cats}, {c: 0 for c in LABELER.cats}
for q, o, e in zip(tr["question_class"], tr["original"], tr["_canon"]):
    for c, _ in (aspect_pairs(q, o) or []):
        if c in asked:
            asked[c] += 1
    for ci, _ in e:
        got[LABELER.cats[ci]] += 1
cov = sum(got.values()) / max(sum(asked.values()), 1)
print(f"\nCakupan label kanonik (train+val): {sum(got.values())}/{sum(asked.values())} = {100*cov:.3f}%")
assert cov >= 0.99, "Cakupan label < 99% — periksa parsing `original`."
print(f"Rata-rata aspek berlabel per baris: {np.mean([len(e) for e in tr['_canon']]):.2f}")
print(f"Indeks kategori polyp_size: {LABELER.size_cat_idx} | Peta lokal 4 tier fisik {POLYP_SIZE_ORDERED_TIERS}: {LABELER.size_rank2local}")

print("\nSelf-test modul (identitas init = M4, Gram-Schmidt |cos|<1e-5, CORN, Router, RNG = M4, fp16 kasus terburuk):")
print(" ", OUC.self_test(CANON_SPECS))

✓ ortho_unified_core ortho-unified-2026-11-15 | as_list lulus uji format asli dataset

Spesifikasi kanonik (min_count=20, min_cover=0.99) | baris tak sejajar: 0
  kategori                   aspek  |V|  cover%  multi  status
  abnormality_color          14997   13  100.00   True  OK
  abnormality_location       15191    9   99.98   True  OK
  abnormality_presence       15013    5   99.96   True  OK
  box_artifact_presence      18954    2  100.00  False  OK
  finding_count              16525    4   99.87  False  OK
  finding_presence            2041    1  100.00  False  jawaban tunggal
  instrument_count           17241    4  100.00  False  OK
  instrument_location        15309   10  100.00   True  OK
  instrument_presence        15404    5   99.94   True  OK
  landmark_color               432    4   97.85   True  cover 0.978<0.99
  landmark_location          10759   10  100.00   True  OK
  landmark_presence           9761    4   99.85  False  OK
  polyp_count                15175    5  

# [A8] Dataset `VQAOrthoDS` & fungsi collate

In [7]:
# [A8] Dataset VQAOrthoDS & fungsi collate (target VQA = prosa murni seperti M4 & B1 + label kanonik)
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from unittest.mock import patch
from transformers import AutoProcessor, AutoModelForCausalLM
from transformers.dynamic_module_utils import get_imports

PROMPT_PREFIX = "<VQA> "

def _skip_flash_proc(f):
    return [i for i in get_imports(f) if i != "flash_attn"]

with patch("transformers.dynamic_module_utils.get_imports", _skip_flash_proc):
    processor = AutoProcessor.from_pretrained(BASE_ID, trust_remote_code=True)

class VQAOrthoDS(Dataset):
    def __init__(self, df):
        self.q = df["question"].astype(str).str.strip().tolist()
        self.a = df["answer"].astype(str).str.strip().tolist()     # TANPA slot: identik M4 & B1
        self.p = [IMG_DIR / f"{i}.jpg" for i in df["_img"]]
        self.c = df["_canon"].tolist()
    def __len__(self):
        return len(self.q)
    def __getitem__(self, i):
        p = self.p[i]
        try:
            img = Image.open(p).convert("RGB")
        except Exception:
            drive_src = BASE / "all_images" / p.name
            if drive_src.exists():
                shutil.copy2(drive_src, p)
                img = Image.open(p).convert("RGB")
            else:
                raise
        return {"q": PROMPT_PREFIX + self.q[i], "a": self.a[i], "img": img, "canon": self.c[i]}

def collate(batch):
    enc = processor(text=[b["q"] for b in batch],
                    images=[b["img"] for b in batch],
                    return_tensors="pt", padding=True)
    lab = processor.tokenizer([b["a"] for b in batch], return_tensors="pt",
                              padding=True, return_token_type_ids=False).input_ids
    lab[lab == processor.tokenizer.pad_token_id] = -100
    cy, cm = LABELER.to_tensors([b["canon"] for b in batch])
    return {
        "input_ids": enc["input_ids"],
        "pixel_values": enc["pixel_values"],
        "attention_mask": enc.get("attention_mask"),
        "labels": lab,
        "canon_y": cy,
        "canon_m": cm,
    }

ds_train = VQAOrthoDS(tr[tr._split == "train"])
ds_val   = VQAOrthoDS(tr[tr._split == "val"])
print(f"Dataset siap | Train: {len(ds_train)} sampel | Val: {len(ds_val)} sampel")

b_test = collate([ds_train[i] for i in range(4)])
bos_id = processor.tokenizer.bos_token_id
assert (b_test["labels"][:, 0] == bos_id).all(), "Token ke-0 pada labels harus bos_token_id!"
assert b_test["canon_y"].shape == (4, LABELER.T) and b_test["canon_m"].shape == (4, LABELER.C)
assert b_test["canon_m"].sum() > 0, "Batch uji tidak berisi label kanonik sama sekali."
dec = processor.tokenizer.batch_decode(b_test["labels"].clamp(min=0), skip_special_tokens=True)
_norm = lambda t: " ".join(str(t).split())
assert [_norm(d) for d in dec] == [_norm(a) for a in ds_train.a[:4]], "Target VQA tidak identik dengan jawaban asli!"
print(f"[VERIFIED] BOS ({bos_id}) | target = prosa asli | canon_y {tuple(b_test['canon_y'].shape)} "
      f"| aspek berlabel di batch uji: {int(b_test['canon_m'].sum())}")
del b_test, dec

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset siap | Train: 129023 sampel | Val: 14571 sampel
[VERIFIED] BOS (0) | target = prosa asli | canon_y (4, 88) | aspek berlabel di batch uji: 11


# [A9] SAKELAR TUNGGAL `VARIANT`

In [8]:
# [A9] SAKELAR TUNGGAL VARIANT — Ortho-Unified Gold Run atau Ablasi
#   "O2_Ortho_Unified_Full" -> GOLD RUN: M4 + OrthoUnifiedBridge lengkap (9D+Fourier + QueryRouter + GramSchmidt + CORN+EMD + PatchFiLM)
#   "O1_Ortho_NoPatchFiLM"  -> Ablasi tanpa Per-Patch Coordinate-Aware FiLM
#   "O0_M4_repro"           -> Kontrol replikasi M4 murni tanpa bridge
VARIANT = "O2_Ortho_Unified_Full"

M4_GEO_KWARGS = dict(use_geope=True, use_diff=True, gate_mode="film")
VARIANTS = {
    "O2_Ortho_Unified_Full": dict(
        use_bridge=True, alpha_canon=0.05,
        use_gram_schmidt=True, use_query_router=True, use_corn_size=True, use_patch_film=True,
        beta_router=0.20, beta_corn=0.30, beta_emd=0.20, beta_div=0.05,
    ),
    "O1_Ortho_NoPatchFiLM": dict(
        use_bridge=True, alpha_canon=0.05,
        use_gram_schmidt=True, use_query_router=True, use_corn_size=True, use_patch_film=False,
        beta_router=0.20, beta_corn=0.30, beta_emd=0.20, beta_div=0.05,
    ),
    "O0_M4_repro": dict(
        use_bridge=False, alpha_canon=0.0,
        use_gram_schmidt=False, use_query_router=False, use_corn_size=False, use_patch_film=False,
        beta_router=0.0, beta_corn=0.0, beta_emd=0.0, beta_div=0.0,
    ),
}
assert VARIANT in VARIANTS, f"VARIANT tidak dikenal: {VARIANT}"
CFG = VARIANTS[VARIANT]
ALPHA_CANON = CFG["alpha_canon"]        # = 0,05 (identik CL-Bridge & B1_M4_CanonBridge)
UNFREEZE_IMG_PROJ = True                # identik M4 & B1_M4_CanonBridge
HEAD_FILE = "ortho_unified_head.pt"     # dipakai [D1] (simpan) dan [E2] (muat)

def _unwrap(m):
    """Cari modul yang mendefinisikan _encode_image (dipakai [A10] dan [E2])."""
    seen = set()
    def walk(x, depth=0):
        if depth > 6 or id(x) in seen: return None
        seen.add(id(x))
        if "_encode_image" in type(x).__dict__: return x
        for name in ("base_model", "model"):
            child = getattr(x, name, None)
            if child is not None and child is not x:
                r = walk(child, depth + 1)
                if r is not None: return r
        return None
    found = walk(m)
    assert found is not None, "could not locate the module defining _encode_image"
    return found

def make_head(d_vis=768):
    return GeoFiLMOrthoUnified(
        d_vis, CANON_SPECS, M4_GEO_KWARGS,
        use_bridge=CFG["use_bridge"],
        use_gram_schmidt=CFG["use_gram_schmidt"],
        use_query_router=CFG["use_query_router"],
        use_corn_size=CFG["use_corn_size"],
        use_patch_film=CFG["use_patch_film"],
    ).to(dev)

print("Active VARIANT :", VARIANT, "|", CFG)
print("Geo (M4)       :", M4_GEO_KWARGS)
print("Kategori bridge:", len(CANON_SPECS), "| atom:", LABELER.T)

Active VARIANT : O2_Ortho_Unified_Full | {'use_bridge': True, 'alpha_canon': 0.05, 'use_gram_schmidt': True, 'use_query_router': True, 'use_corn_size': True, 'use_patch_film': True, 'beta_router': 0.2, 'beta_corn': 0.3, 'beta_emd': 0.2, 'beta_div': 0.05}
Geo (M4)       : {'use_geope': True, 'use_diff': True, 'gate_mode': 'film'}
Kategori bridge: 16 | atom: 88


# [A10] Build model Florence-2 + LoRA r=32 + hook

In [9]:
# [A10] Build model Florence-2 + LoRA r=32 + pasang hook get_input_embeddings & _encode_image
from peft import LoraConfig, get_peft_model
import transformers.dynamic_module_utils as dmu
orig_get_imports = dmu.get_imports

def _skip_flash(name, *args, **kwargs):
    if "flash_attn" in str(name):
        return None
    return orig_get_imports(name, *args, **kwargs)

LORA_R     = 32
LORA_ALPHA = LORA_R * 2

def build_model():
    with patch("transformers.dynamic_module_utils.get_imports", _skip_flash):
        m = AutoModelForCausalLM.from_pretrained(
            BASE_ID, trust_remote_code=True, torch_dtype=torch.float32)
    cfg = LoraConfig(
        r=LORA_R, lora_alpha=LORA_ALPHA,
        lora_dropout=lora_cfg["lora_dropout"],
        target_modules=list(lora_cfg["target_modules"]),
        bias=lora_cfg.get("bias", "none"),
        task_type=lora_cfg.get("task_type"))
    m = get_peft_model(m, cfg)

    core = _unwrap(m)
    if UNFREEZE_IMG_PROJ:
        core.image_projection.requires_grad_(True)
        for p in core.image_proj_norm.parameters():
            p.requires_grad_(True)

    cache = {"query_embeds": None, "query_mask": None, "reg_loss": None, "canon_logits": None, "ortho_aux": None}
    head_holder = [None]

    emb_layer = core.get_input_embeddings()
    def _emb_hook(module, args, output):
        if torch.is_tensor(output) and output.dim() == 3 and output.shape[1] > 2:
            cache["query_embeds"] = output.detach()
    emb_hook_handle = emb_layer.register_forward_hook(_emb_hook)

    orig_encode = core._encode_image
    def wrapped_encode_image(pixel_values):
        out = orig_encode(pixel_values)
        h_mod = head_holder[0]
        if h_mod is not None:
            bridged, reg_loss, canon_logits = h_mod.forward_bridge(
                out, query_embeds=cache.get("query_embeds"), query_mask=cache.get("query_mask"))
            cache["feats"] = out
            cache["reg_loss"] = reg_loss
            cache["canon_logits"] = canon_logits
            cache["ortho_aux"] = h_mod.latest_aux
            return bridged.to(out.dtype)
        cache["feats"] = out
        cache["reg_loss"] = torch.tensor(0.0, device=out.device)
        cache["canon_logits"] = None
        cache["ortho_aux"] = None
        return out

    core._encode_image = wrapped_encode_image
    m._feat_cache = cache
    object.__setattr__(m, "_core_ref", [core])
    object.__setattr__(m, "_head_ref", head_holder)
    object.__setattr__(m, "_emb_hook_ref", [emb_hook_handle])
    return m.to(dev)

model = build_model()
core = model._core_ref[0]
D_VIS = core.image_projection.shape[1]
head = make_head(D_VIS)
model._head_ref[0] = head

model.print_trainable_parameters()
n_geo = sum(p.numel() for p in head.geo.parameters())
n_br  = sum(p.numel() for p in head.bridge.parameters()) if head.bridge is not None else 0
print(f"Kepala GeoFiLM (M4)   : {n_geo:,} param")
print(f"OrthoUnifiedBridge    : {n_br:,} param  (+{100*n_br/max(n_geo,1):.1f}% dari kepala M4)")

REG_WEIGHT = 0.02
LR_GROUPS  = {"lora": HP["lr"], "head": HP["lr"] * 50, "bridge": HP["lr"] * 100,
              "head_lr_mult": 50, "bridge_lr_mult": 100}
print("REG_WEIGHT:", REG_WEIGHT, "| ALPHA_CANON:", ALPHA_CANON, "| LR_GROUPS:", LR_GROUPS)

W1116 05:24:22.292000 81307 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


trainable params: 5,191,776 || all params: 235,817,824 || trainable%: 2.2016
Kepala GeoFiLM (M4)   : 1,730,850 param
OrthoUnifiedBridge    : 1,655,273 param  (+95.6% dari kepala M4)
REG_WEIGHT: 0.02 | ALPHA_CANON: 0.05 | LR_GROUPS: {'lora': 1e-05, 'head': 0.0005, 'bridge': 0.001, 'head_lr_mult': 50, 'bridge_lr_mult': 100}


# SMOKE TEST — Uji Pra-Training (Identitas Step 0 == M4, Ortogonalitas Gram-Schmidt, Router, CORN)
## [A11] Smoke Test Pra-Training

In [ ]:
# [A11] Smoke Test Pra-Training — urutan [D2] persis (scale->backward->unscale->clip->step->update)
print("=" * 78)
print(f" SMOKE TEST PRA-TRAINING | VARIANT = {VARIANT}")
print("=" * 78)

SMOKE_STEPS, SMOKE_BS = 3, 2
model.train(); head.train()
model.zero_grad(set_to_none=True); head.zero_grad(set_to_none=True)
model._feat_cache.clear()
use_fp16 = HP.get("fp16", True) and (dev == "cuda")
smk_scaler = torch.amp.GradScaler("cuda", enabled=use_fp16)
smk_opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad] + list(head.parameters()), lr=1e-6)

steps_applied, identity_ok = 0, None
for st in range(SMOKE_STEPS):
    bb = collate([ds_train[st * SMOKE_BS + j] for j in range(SMOKE_BS)])
    bb = {k: (v.to(dev, non_blocking=True) if torch.is_tensor(v) else v) for k, v in bb.items()}
    model._feat_cache["query_mask"] = bb["attention_mask"]
    with torch.amp.autocast("cuda", enabled=use_fp16):
        o = model(input_ids=bb["input_ids"], pixel_values=bb["pixel_values"],
                  attention_mask=bb["attention_mask"], labels=bb["labels"])
        rl = model._feat_cache.get("reg_loss")
        lg = model._feat_cache.get("canon_logits")
        ox = model._feat_cache.get("ortho_aux")
        if lg is not None:
            lc, n_lab, st_aux = ortho_unified_loss(
                lg, ox, bb["canon_y"], bb["canon_m"], LABELER,
                beta_router=CFG["beta_router"], beta_corn=CFG["beta_corn"],
                beta_emd=CFG["beta_emd"], beta_div=CFG["beta_div"]
            )
        else:
            lc, n_lab, st_aux = torch.tensor(0.0, device=dev), 0, {}
        tot = o.loss + REG_WEIGHT * rl + ALPHA_CANON * lc
    assert torch.isfinite(tot), f"Loss NaN/Inf pada step {st} — BUG model."
    assert model._feat_cache.get("query_embeds") is not None, "Hook query embedding tidak terpanggil!"
    if CFG["use_bridge"]:
        assert lg is not None and lg.shape == (SMOKE_BS, LABELER.T), f"canon_logits {None if lg is None else lg.shape}"

    if st == 0:
        with torch.no_grad():
            f0 = model._feat_cache["feats"]
            qe, qm = model._feat_cache.get("query_embeds"), model._feat_cache.get("query_mask")
            b_out, _, _ = head.forward_bridge(f0, qe, qm)
            g_out, _ = head.geo.forward_bridge(f0, qe, qm)
            identity_ok = bool(torch.equal(b_out, g_out))
        assert identity_ok, "Step 0 tidak identik dengan M4 — gerbang bridge tidak nol?"

    sc0 = smk_scaler.get_scale()
    smk_scaler.scale(tot).backward()
    if CFG["use_bridge"]:
        assert head.bridge.gate_global.grad is not None and head.bridge.fc[0].weight.grad is not None
        assert head.bridge.router_mlp[0].weight.grad is not None
    smk_scaler.unscale_(smk_opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    torch.nn.utils.clip_grad_norm_(head.geo.parameters(), max_norm=1.0)
    if head.bridge is not None:
        torch.nn.utils.clip_grad_norm_(head.bridge.parameters(), max_norm=1.0)
    smk_scaler.step(smk_opt); smk_scaler.update()
    applied = smk_scaler.get_scale() >= sc0
    steps_applied += int(applied)
    print(f"  step {st}: L_VQA {float(o.loss.detach()):.4f} | L_struct {float(rl.detach()):.4f} | "
          f"L_ortho {float(lc.detach()):.4f} (router={st_aux.get('L_router',0):.3f}, corn={st_aux.get('L_corn',0):.3f}) | "
          f"{'DITERAPKAN' if applied else 'dilewati (overflow fp16 awal, normal)'}")
    smk_opt.zero_grad(set_to_none=True); model.zero_grad(set_to_none=True); head.zero_grad(set_to_none=True)
    model._feat_cache.clear()
    del o, rl, lg, ox, lc, tot, bb

assert steps_applied > 0, "Tidak ada optimizer step yang diterapkan!"
print(f" ✓ Identitas step 0 = M4 : {identity_ok}")
print(f" ✓ Trainability          : {steps_applied}/{SMOKE_STEPS} step diterapkan")
del smk_opt, smk_scaler
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("=" * 78)
print(" ✓ SMOKE TEST LULUS. Lanjut ke [B1] (model dibangun ulang bersih di sana).")
print("=" * 78)

# PART B — FRESH RUN SETUP (Jalankan untuk Memulai Run Baru dari Step 0)
<a id="part-b"></a>
## [B1] Reseed + rebuild model

In [ ]:
# [B1] Reseed + rebuild model bersih untuk run FRESH (urutan RNG identik run M4 & B1)
from datetime import datetime

if 'model' in globals() and model is not None:
    model.zero_grad(set_to_none=True)
    if hasattr(model, '_feat_cache'):
        model._feat_cache.clear()
    if hasattr(model, '_emb_hook_ref'):
        for h in model._emb_hook_ref:
            try: h.remove()
            except Exception: pass
if 'head' in globals() and head is not None:
    head.zero_grad(set_to_none=True)
model = head = None
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

model = build_model()
core = model._core_ref[0]
D_VIS = core.image_projection.shape[1]
head = make_head(D_VIS)          # OrthoUnifiedBridge memulihkan RNG -> init M4 & urutan batch = run M4 & B1
model._head_ref[0] = head

total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 16.0
EFFECTIVE_BS = HP["batch_size"] * HP["grad_accum"]
MICRO_BS     = 16 if (torch.cuda.is_available() and total_vram >= 30.0) else 4
GRAD_ACCUM   = EFFECTIVE_BS // MICRO_BS
assert MICRO_BS * GRAD_ACCUM == EFFECTIVE_BS
if MICRO_BS != 16:
    print("PERINGATAN: run M4 & B1 memakai micro-batch 16. Gunakan GPU >=30GB (A100/L4) agar kurva val sebanding.")

RUN_TAG  = datetime.now().strftime("%Y%m%d-%H%M%S")
RUN_NAME = f"{RUN_TAG}_{VARIANT}_r{LORA_R}_s{SEED}"
RUN_DIR  = OUTDIR / RUN_NAME
(RUN_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(RUN_DIR / "history").mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_name": RUN_NAME,
    "started_at": datetime.now().isoformat(timespec="seconds"),
    "variant": VARIANT,
    "variant_cfg": CFG,
    "geo_kwargs": M4_GEO_KWARGS,
    "core_version": OUC.CORE_VERSION,
    "canon_min_count": CANON_MIN_COUNT,
    "canon_min_cover": CANON_MIN_COVER,
    "canon_specs": CANON_SPECS,
    "alpha_canon": ALPHA_CANON,
    "seed": SEED,
    "reg_weight": REG_WEIGHT,
    "lr_groups": LR_GROUPS,
    "hp": HP,
    "effective_batch": EFFECTIVE_BS,
    "micro_batch": MICRO_BS,
    "grad_accum": GRAD_ACCUM,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "n_head_params": int(sum(p.numel() for p in head.parameters())),
    "unfreeze_image_projection": UNFREEZE_IMG_PROJ,
    "loss_vqa": "cross_entropy (out.loss bawaan, tanpa focal)",
    "target": "prosa asli (tanpa slot)",
}
(RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
(RUN_DIR / "canon_specs.json").write_text(json.dumps(CANON_SPECS, indent=1))
print("✓ Fresh model built & RUN_DIR created ->", RUN_DIR)
print(f"  Effective batch = {EFFECTIVE_BS} (micro {MICRO_BS} x accum {GRAD_ACCUM})")

# PART C — RESUME ONLY (Jalankan HANYA Jika Training Terputus di Tengah Jalan)
<a id="part-c"></a>
## [C1] Cari & arahkan ke folder run yang terputus

In [10]:
# [C1] Cari & arahkan ke folder run yang terputus sesuai VARIANT
cands = []
for p in sorted(OUTDIR.glob("*")):
    if p.is_dir() and (p / "resume/state.pt").exists() and (p / "resume/adapter_model.safetensors").exists():
        d = torch.load(p / "resume/state.pt", map_location="cpu", weights_only=False)
        cands.append(p)
        print(p.name, "| step", d["global_step"], "| epoch", d["epoch"])
assert cands, f"Tidak ada run dengan resume slot di {OUTDIR}"

match = [p for p in cands if VARIANT in p.name]
RUN_DIR = match[-1] if match else cands[-1]
CONFIG = json.loads((RUN_DIR / "config.json").read_text())
assert CONFIG.get("core_version") == OUC.CORE_VERSION, \
    f"Run dilatih dengan core {CONFIG.get('core_version')}, modul sekarang {OUC.CORE_VERSION}"
assert list(CONFIG["canon_specs"].items()) == list(json.loads(json.dumps(CANON_SPECS)).items()), \
    "Spesifikasi kanonik berbeda dari run yang di-resume."
MICRO_BS, GRAD_ACCUM = CONFIG["micro_batch"], CONFIG["grad_accum"]
print("\nRESUME TARGET ->", RUN_DIR.name)

20261116-000618_O2_Ortho_Unified_Full_r32_s42 | step 16128 | epoch 4

RESUME TARGET -> 20261116-000618_O2_Ortho_Unified_Full_r32_s42


# PART D — TRAINING LOOP UTAMA (5 Epoch, Live Pembanding vs `M4` & `B1_M4_CanonBridge`)
<a id="part-d"></a>

Setiap validasi mencetak:
- Selisih `val_L_VQA` terhadap **`M4_GeoFiLM`** DAN terhadap **`B1_M4_CanonBridge`** pada step yang sama (step 0 harus $= 5,0737$ dengan $\Delta = +0,0000$).
- Akurasi rata-rata 16 kepala kanonik (`canon acc`), akurasi 3 kepala lokasi (`loc3 acc`), akurasi kepala CORN `polyp_size`, F1 Query-Aspect Router, serta verifikasi ortogonalitas Gram-Schmidt ($|\cos| < 10^{-5}$).

## [D1] Setup DataLoaders, Optimizer, Scheduler, Checkpointing, & Validasi

In [11]:
# [D1] Setup DataLoaders, Optimizer, Scheduler, Checkpointing, & Validasi (vs M4 & B1_M4_CanonBridge)
from transformers import get_scheduler
from tqdm.auto import tqdm
import collections
from datetime import datetime

TRAIN_LOG  = RUN_DIR / "history/train.jsonl"
VAL_LOG    = RUN_DIR / "history/val.jsonl"
CKPT_DIR   = RUN_DIR / "checkpoints"
RESUME_DIR = RUN_DIR / "resume"
RESUME_DIR.mkdir(parents=True, exist_ok=True)

def log_line(path, d):
    with open(path, "a") as f:
        f.write(json.dumps(d) + "\n")

def save_ckpt(step, tag=None):
    td = CKPT_DIR / (tag if tag else f"step-{step:06d}")
    td.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(str(td))
    torch.save(head.state_dict(), td / HEAD_FILE)
    (td / "canon_specs.json").write_text(json.dumps(CANON_SPECS, indent=1))
    core = model._core_ref[0]
    torch.save({"image_projection": core.image_projection.data.clone(),
                "image_proj_norm":  core.image_proj_norm.state_dict()}, td / "image_proj.pt")
    print(f"saved checkpoint -> {td.name}")

def save_resume(step, epoch):
    tmp = RESUME_DIR / "state.pt.tmp"
    torch.save({
        "global_step": step, "epoch": epoch,
        "optimizer": opt.state_dict(), "scheduler": sched.state_dict(), "scaler": scaler.state_dict(),
        "rng": {"python": random.getstate(), "numpy": np.random.get_state(),
                "torch": torch.get_rng_state(),
                "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None},
        "head_state": head.state_dict(),
        "saved_at": datetime.now().isoformat(timespec="seconds"),
    }, tmp)
    os.replace(tmp, RESUME_DIR / "state.pt")
    model.save_pretrained(str(RESUME_DIR))
    core = model._core_ref[0]
    torch.save({"image_projection": core.image_projection.data.clone(),
                "image_proj_norm":  core.image_proj_norm.state_dict()}, RESUME_DIR / "image_proj.pt")

dl_tr = DataLoader(ds_train, batch_size=MICRO_BS, shuffle=True,
                   collate_fn=collate, num_workers=4 if IS_COLAB else 0, pin_memory=True)
dl_va = DataLoader(ds_val, batch_size=MICRO_BS, shuffle=False,
                   collate_fn=collate, num_workers=2 if IS_COLAB else 0)

_groups = [
    {"params": [p for p in model.parameters() if p.requires_grad],
     "lr": LR_GROUPS["lora"], "weight_decay": HP["weight_decay"]},
    {"params": list(head.geo.parameters()),
     "lr": LR_GROUPS["head"], "weight_decay": HP["weight_decay"]},
]
if head.bridge is not None:
    _groups.append({"params": list(head.bridge.parameters()),
                    "lr": LR_GROUPS["bridge"], "weight_decay": HP["weight_decay"]})
opt = torch.optim.AdamW(_groups)
_n_opt = sum(p.numel() for g in opt.param_groups for p in g["params"])
_n_all = sum(p.numel() for p in model.parameters() if p.requires_grad) + sum(p.numel() for p in head.parameters())
assert _n_opt == _n_all, f"Optimizer tidak mencakup semua parameter terlatih ({_n_opt} vs {_n_all})"

STEPS_PER_EPOCH = len(dl_tr) // GRAD_ACCUM
TOTAL_STEPS     = int(STEPS_PER_EPOCH * HP["epochs"])
WARMUP_STEPS    = int(TOTAL_STEPS * HP["warmup_ratio"])
print(f"steps/epoch: {STEPS_PER_EPOCH} | total: {TOTAL_STEPS} | warmup: {WARMUP_STEPS}")

sched = get_scheduler(HP["scheduler"], optimizer=opt,
                      num_warmup_steps=WARMUP_STEPS, num_training_steps=TOTAL_STEPS)
scaler = torch.amp.GradScaler("cuda", enabled=HP["fp16"] and (dev == "cuda"))

def clip_all():
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    torch.nn.utils.clip_grad_norm_(head.geo.parameters(), max_norm=1.0)
    if head.bridge is not None:
        torch.nn.utils.clip_grad_norm_(head.bridge.parameters(), max_norm=1.0)

# Baca kurva val M4 & B1_M4_CanonBridge secara LIVE
M4_VAL, B1_VAL = {}, {}
_m4 = [p for p in sorted(M4_DIR.glob("*M4_GeoFiLM*")) if (p / "history/val.jsonl").exists()] if M4_DIR.exists() else []
if _m4:
    for line in (_m4[-1] / "history/val.jsonl").read_text().splitlines():
        r = json.loads(line)
        M4_VAL[r["global_step"]] = r["val_vqa_loss"]
_b1 = [p for p in sorted(B1_DIR.glob("*B1_M4_CanonBridge*")) if (p / "history/val.jsonl").exists()] if B1_DIR.exists() else []
if _b1:
    for line in (_b1[-1] / "history/val.jsonl").read_text().splitlines():
        r = json.loads(line)
        B1_VAL[r["global_step"]] = r["val_vqa_loss"]
print(f"Pembanding kurva val -> M4: {len(M4_VAL)} titik | B1_CanonBridge: {len(B1_VAL)} titik")

LOC3_CATS = {"abnormality_location", "instrument_location", "landmark_location"}

def validate_vqa(step, epoch, max_batches=60):
    model.eval(); head.eval()
    v_losses, r_losses, c_losses = [], [], []
    cstat = collections.defaultdict(lambda: [0, 0])
    r_tp, r_fp, r_fn = 0, 0, 0
    corn_ok, corn_n = 0, 0
    cos_g_list, cos_s_list = [], []

    with torch.no_grad():
        for k, bb in enumerate(dl_va):
            if max_batches and k >= max_batches:
                break
            bb = {kk: (v.to(dev, non_blocking=True) if torch.is_tensor(v) else v) for kk, v in bb.items()}
            model._feat_cache["query_mask"] = bb.get("attention_mask")
            with torch.amp.autocast("cuda", enabled=HP["fp16"] and (dev == "cuda")):
                o = model(input_ids=bb["input_ids"], pixel_values=bb["pixel_values"],
                          attention_mask=bb.get("attention_mask"), labels=bb["labels"])
                rl = model._feat_cache.get("reg_loss")
                lg = model._feat_cache.get("canon_logits")
                ox = model._feat_cache.get("ortho_aux")
            v_losses.append(float(o.loss.item()))
            if rl is not None:
                r_losses.append(float(rl.item()))
            if lg is not None:
                lc, _, st_aux = ortho_unified_loss(
                    lg, ox, bb["canon_y"], bb["canon_m"], LABELER,
                    beta_router=CFG["beta_router"], beta_corn=CFG["beta_corn"],
                    beta_emd=CFG["beta_emd"], beta_div=CFG["beta_div"]
                )
                c_losses.append(float(lc))
                for c, (n, ok) in canonical_batch_stats(lg, bb["canon_y"], bb["canon_m"], LABELER).items():
                    cstat[c][0] += n; cstat[c][1] += ok
                if ox is not None:
                    cos_g_list.append(st_aux["cos_glob"])
                    cos_s_list.append(st_aux["cos_spat"])
                    rp = (ox["router_probs"].float() >= 0.5).float()
                    mg = bb["canon_m"].float()
                    r_tp += int((rp * mg).sum().item())
                    r_fp += int((rp * (1.0 - mg)).sum().item())
                    r_fn += int(((1.0 - rp) * mg).sum().item())
                    ord_y, ord_m = LABELER.extract_size_ordinal_targets(bb["canon_y"], bb["canon_m"])
                    if (ord_m > 0.5).any():
                        pr_rk = (ox["corn_p_cum"].float() >= 0.5).sum(dim=-1)
                        corn_ok += int((pr_rk[ord_m > 0.5] == ord_y[ord_m > 0.5]).sum().item())
                        corn_n  += int((ord_m > 0.5).sum().item())
            model._feat_cache.clear()
    model.train(); head.train()
    gg, gs = head.gate_mean_abs()
    g_pf = round(float(torch.tanh(head.bridge.gate_patch_film).item()), 4) if head.bridge is not None else 0.0
    acc_macro = (float(np.mean([ok / n for n, ok in cstat.values() if n])) if cstat else None)
    loc3_vals = [cstat[c][1] / cstat[c][0] for c in LOC3_CATS if c in cstat and cstat[c][0] > 0]
    acc_loc3  = float(np.mean(loc3_vals)) if loc3_vals else None
    router_f1 = (2.0 * r_tp) / max(2.0 * r_tp + r_fp + r_fn, 1.0) if (r_tp + r_fn) > 0 else None
    corn_acc  = (corn_ok / corn_n) if corn_n > 0 else None

    vl = round(float(np.mean(v_losses)), 4) if v_losses else None
    m4 = M4_VAL.get(step)
    b1 = B1_VAL.get(step)
    rec = {
        "global_step": step, "epoch": epoch,
        "val_vqa_loss": vl,
        "val_reg_loss": round(float(np.mean(r_losses)), 4) if r_losses else None,
        "val_ortho_loss": round(float(np.mean(c_losses)), 4) if c_losses else None,
        "val_canon_acc_macro": round(acc_macro, 4) if acc_macro is not None else None,
        "val_loc3_acc_macro": round(acc_loc3, 4) if acc_loc3 is not None else None,
        "val_router_f1": round(router_f1, 4) if router_f1 is not None else None,
        "val_corn_size_acc": round(corn_acc, 4) if corn_acc is not None else None,
        "m4_val_vqa_loss": m4,
        "b1_val_vqa_loss": b1,
        "gate_global_absmean": gg, "gate_spatial_absmean": gs, "gate_patch_film": g_pf,
        "cos_glob_after": float(np.mean(cos_g_list)) if cos_g_list else 0.0,
        "cos_spat_after": float(np.mean(cos_s_list)) if cos_s_list else 0.0,
        "pe_scale": round(float(torch.tanh(head.pe_scale).item()), 4),
        "at": datetime.now().isoformat(timespec="seconds"),
    }
    log_line(VAL_LOG, rec)
    vs_m4 = (f" | M4 {m4:.4f} (Δ {vl - m4:+.4f})" if m4 is not None else "")
    vs_b1 = (f" | B1 {b1:.4f} (Δ {vl - b1:+.4f})" if b1 is not None else "")
    ca    = (f" | canon {100*acc_macro:.1f}%" if acc_macro is not None else "")
    la    = (f" (loc3 {100*acc_loc3:.1f}%)" if acc_loc3 is not None else "")
    rf    = (f" | routerF1 {100*router_f1:.1f}%" if router_f1 is not None else "")
    print(f"\n[VAL step {step} ep {epoch}] val_L_VQA={vl:.4f}{vs_m4}{vs_b1}{ca}{la}{rf} "
          f"| gate |g| {gg:.4f}/{gs:.4f}/pf:{g_pf:+.3f}")
    if step == 0 and m4 is not None and abs(vl - m4) > 0.02:
        print("  PERINGATAN: val step 0 berbeda dari M4 > 0,02 — periksa micro-batch / split.")
    return rec

steps/epoch: 4032 | total: 20160 | warmup: 2016
Pembanding kurva val -> M4: 46 titik | B1_CanonBridge: 46 titik


## [D2] Jalankan Loop Training 5 Epoch

In [12]:
# [D2] Jalankan Loop Training 5 Epoch (dengan Auto-Restore saat Resume & penjaga divergensi)
LOG_EVERY  = 50
VAL_EVERY  = 500
SAVE_EVERY = 500

gstep = 0
start_epoch = 0

resume_state = RUN_DIR / "resume/state.pt"
if resume_state.exists():
    d = torch.load(resume_state, map_location="cpu", weights_only=False)
    gstep = d["global_step"]
    start_epoch = d["epoch"]
    opt.load_state_dict(d["optimizer"])
    sched.load_state_dict(d["scheduler"])
    scaler.load_state_dict(d["scaler"])
    random.setstate(d["rng"]["python"])
    np.random.set_state(d["rng"]["numpy"])
    torch.set_rng_state(d["rng"]["torch"])
    if torch.cuda.is_available() and d["rng"]["cuda"] is not None:
        torch.cuda.set_rng_state_all(d["rng"]["cuda"])
    head.load_state_dict(d["head_state"])
    from peft import set_peft_model_state_dict
    from safetensors.torch import load_file
    set_peft_model_state_dict(model, load_file(str(RUN_DIR / "resume/adapter_model.safetensors")))
    if (RUN_DIR / "resume/image_proj.pt").exists():
        ip_st = torch.load(RUN_DIR / "resume/image_proj.pt", map_location=dev)
        core = model._core_ref[0]
        core.image_projection.data.copy_(ip_st["image_projection"])
        core.image_proj_norm.load_state_dict(ip_st["image_proj_norm"])
    print(f"✓ Resumed dari step {gstep}, epoch {start_epoch}")

t0 = time.time()
validate_vqa(gstep, start_epoch, max_batches=40)       # jadwal max_batches identik M4 & B1 (40/50/100)

DIVERGED = False
for ep in range(start_epoch, int(math.ceil(HP["epochs"]))):
    if DIVERGED:
        break
    model.train(); head.train()
    pbar = tqdm(dl_tr, desc=f"{VARIANT} epoch {ep}")
    for i, bb in enumerate(pbar):
        bb = {k: (v.to(dev, non_blocking=True) if torch.is_tensor(v) else v) for k, v in bb.items()}
        model._feat_cache["query_mask"] = bb.get("attention_mask")
        with torch.amp.autocast("cuda", enabled=HP["fp16"] and (dev == "cuda")):
            o = model(input_ids=bb["input_ids"], pixel_values=bb["pixel_values"],
                      attention_mask=bb.get("attention_mask"), labels=bb["labels"])
            rl = model._feat_cache.get("reg_loss")
            if rl is None:
                rl = torch.tensor(0.0, device=dev)
            lg = model._feat_cache.get("canon_logits")
            ox = model._feat_cache.get("ortho_aux")
            if lg is not None:
                lc, n_lab, st_aux = ortho_unified_loss(
                    lg, ox, bb["canon_y"], bb["canon_m"], LABELER,
                    beta_router=CFG["beta_router"], beta_corn=CFG["beta_corn"],
                    beta_emd=CFG["beta_emd"], beta_div=CFG["beta_div"]
                )
            else:
                lc, n_lab, st_aux = torch.tensor(0.0, device=dev), 0, {}
            loss = (o.loss + REG_WEIGHT * rl + ALPHA_CANON * lc) / GRAD_ACCUM

        scaler.scale(loss).backward()
        model._feat_cache.clear()

        if (i + 1) % GRAD_ACCUM == 0:
            scaler.unscale_(opt)
            clip_all()
            scaler.step(opt)
            scaler.update()
            opt.zero_grad(set_to_none=True); sched.step(); gstep += 1

            if gstep % LOG_EVERY == 0:
                lv = float(o.loss.item())
                if not np.isfinite(lv):
                    print(f"\n!! L_VQA = {lv} pada step {gstep} — dihentikan.")
                    DIVERGED = True
                    break
                gg, gs = head.gate_mean_abs()
                log_line(TRAIN_LOG, {
                    "global_step": gstep, "epoch": ep,
                    "L_vqa": lv, "L_struct": float(rl.item()), "L_ortho": float(lc.detach().item()),
                    **st_aux,
                    "n_canon": int(n_lab), "gate_g": gg, "gate_s": gs,
                    "pe_scale": round(float(torch.tanh(head.pe_scale).item()), 4),
                    "gpu_gb": round(torch.cuda.memory_allocated() / 1e9, 2) if torch.cuda.is_available() else 0.0,
                    "elapsed_s": round(time.time() - t0, 1)
                })
                pbar.set_postfix(vqa=f"{lv:.3f}", ortho=f"{float(lc.detach()):.3f}", g=f"{gg:.3f}")

            if gstep % VAL_EVERY == 0:
                validate_vqa(gstep, ep, max_batches=50)
            if gstep % SAVE_EVERY == 0:
                save_ckpt(gstep); save_resume(gstep, ep)

    if DIVERGED:
        break
    validate_vqa(gstep, ep, max_batches=100)
    save_ckpt(gstep, tag=f"epoch-{ep:02d}")
    save_resume(gstep, ep + 1)

if DIVERGED:
    CONFIG["status"] = "ABORTED - jangan dipakai"
    CONFIG["diverged_at_step"] = gstep
    (RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
    print("RUN DIHENTIKAN. Checkpoint `final` tidak ditulis.")
else:
    save_ckpt(gstep, tag="final")
    CONFIG["finished_at"] = datetime.now().isoformat(timespec="seconds")
    CONFIG["total_steps"] = gstep
    CONFIG["wall_clock_hours"] = round((time.time() - t0) / 3600, 2)
    CONFIG["status"] = "ok"
    (RUN_DIR / "config.json").write_text(json.dumps(CONFIG, indent=2))
    print(f"\n✓ Training {VARIANT} selesai dalam {CONFIG['wall_clock_hours']} jam -> {RUN_DIR}")

✓ Resumed dari step 16128, epoch 4

[VAL step 16128 ep 4] val_L_VQA=0.7372 | M4 0.7593 (Δ -0.0221) | B1 0.7487 (Δ -0.0115) | canon 83.0% (loc3 59.0%) | routerF1 16.6% | gate |g| 0.1230/0.2423/pf:+0.000


/content/drive/MyDrive/DOKTORAL/KvasirVQAX1/GeoFiLM_Paper2/ortho_unified_core.py:907: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  round(float(torch.tanh(b.gate_global).abs().mean()), 5),


O2_Ortho_Unified_Full epoch 4:   0%|          | 0/8064 [00:00<?, ?it/s]


[VAL step 16500 ep 4] val_L_VQA=0.7336 | M4 0.7434 (Δ -0.0098) | B1 0.7327 (Δ +0.0009) | canon 82.3% (loc3 58.0%) | routerF1 14.8% | gate |g| 0.1232/0.2422/pf:+0.000
saved checkpoint -> step-016500

[VAL step 17000 ep 4] val_L_VQA=0.7322 | M4 0.7404 (Δ -0.0082) | B1 0.7317 (Δ +0.0005) | canon 82.7% (loc3 59.0%) | routerF1 16.3% | gate |g| 0.1228/0.2430/pf:+0.000
saved checkpoint -> step-017000

[VAL step 17500 ep 4] val_L_VQA=0.7248 | M4 0.7380 (Δ -0.0132) | B1 0.7307 (Δ -0.0059) | canon 82.7% (loc3 60.4%) | routerF1 15.5% | gate |g| 0.1232/0.2435/pf:+0.000
saved checkpoint -> step-017500

[VAL step 18000 ep 4] val_L_VQA=0.7306 | M4 0.7382 (Δ -0.0076) | B1 0.7303 (Δ +0.0003) | canon 82.2% (loc3 58.5%) | routerF1 15.3% | gate |g| 0.1231/0.2442/pf:+0.000
saved checkpoint -> step-018000

[VAL step 18500 ep 4] val_L_VQA=0.7253 | M4 0.7340 (Δ -0.0087) | B1 0.7249 (Δ +0.0004) | canon 82.4% (loc3 59.2%) | routerF1 15.4% | gate |g| 0.1229/0.2440/pf:+0.000
saved checkpoint -> step-018500

[VAL

# PART E — INFERENSI GENERATIF 15.955 BARIS TEST SET
<a id="part-e"></a>

Inferensi **identik notebook 01 & 05 `[E2]`** (model & kepala fp16, greedy, `max_new_tokens=64`, `attention_mask` TIDAK dikirim ke `.generate()`). Menyimpan:
- `eval/predictions.json` (`{idx: prosa}` — langsung kompatibel dengan `07_Ortho_Unified_Evaluation.ipynb` dan `02_GeoFiLM_Dual_LLM_Judge_Evaluation.ipynb`)
- `eval/canon_preds.json` (`{idx: {kategori: [atom]}}` — prediksi 16 kepala kanonik + CORN)
- `eval/ortho_diag.json` (prediksi Query-Aspect Router $r_c(Q)$, tier CORN `polyp_size`, dan nilai kosinus ortogonalitas Gram-Schmidt).

> **Evaluasi Lengkap Terpisah:** Setelah `[E2]` selesai, buka **`07_Ortho_Unified_Evaluation.ipynb`** untuk menjalankan seluruh metrik evaluasi (Token-F1, 9-Grid Spatial Jaccard/Hedge, Ordinal Polyp Size QWK/MAE, Diagnostik Kepala vs B1, McNemar, dan Dual-LLM Judge `Qwen2.5-7B` + `Llama-3.1-8B`).

## [E1] Muat Test Set & pilih run

In [13]:
# [E1] Muat Test Set (15.955 QA pairs) & Pilih Folder Run yang Akan Dievaluasi
from datasets import load_from_disk
te = load_from_disk(str(BASE / "test")).to_pandas()
print(f"✓ Total test rows loaded: {len(te)}")
assert len(te) == 15955

eval_cands = [p for p in sorted(OUTDIR.glob("*")) if p.is_dir() and (p / "checkpoints/final").exists()]
match_eval = [p for p in eval_cands if VARIANT in p.name]
TARGET_RUN = match_eval[-1] if match_eval else (eval_cands[-1] if eval_cands else globals().get("RUN_DIR", None))
assert TARGET_RUN is not None, f"Tidak ada run dengan checkpoints/final di {OUTDIR}"
print(f"✓ TARGET_RUN untuk inferensi: {TARGET_RUN}")
EVAL_DIR = TARGET_RUN / "eval"
EVAL_DIR.mkdir(parents=True, exist_ok=True)

✓ Total test rows loaded: 15955
✓ TARGET_RUN untuk inferensi: /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper3_Ortho_Unified_r32/20261116-000618_O2_Ortho_Unified_Full_r32_s42


## [E2] Load checkpoint final & jalankan inferensi

In [14]:
# [E2] Load Checkpoint Final (LoRA + GeoFiLMOrthoUnified + image_projection) & Jalankan Inferensi
from peft import PeftModel
from tqdm.auto import tqdm
import transformers.dynamic_module_utils as dmu
orig_get_imports = dmu.get_imports

GEN_CFG = {"max_new_tokens": 64}           # identik notebook 01 & 05 (greedy)

def _skip_flash_e(name, *args, **kwargs):
    if "flash_attn" in str(name): return None
    return orig_get_imports(name, *args, **kwargs)

def load_ortho_for_eval(run_dir):
    cfg = json.loads((run_dir / "config.json").read_text())
    ckpt = run_dir / "checkpoints/final"
    specs = cfg["canon_specs"]
    assert cfg.get("core_version") == OUC.CORE_VERSION, \
        f"Run dilatih dengan core {cfg.get('core_version')}, modul sekarang {OUC.CORE_VERSION}"
    with patch("transformers.dynamic_module_utils.get_imports", _skip_flash_e):
        proc = AutoProcessor.from_pretrained(BASE_ID, trust_remote_code=True)
        base = AutoModelForCausalLM.from_pretrained(BASE_ID, trust_remote_code=True, torch_dtype=torch.float16)
    m = PeftModel.from_pretrained(base, str(ckpt)).to(dev).eval()
    core_e = _unwrap(m)
    ip_st = torch.load(ckpt / "image_proj.pt", map_location=dev)
    core_e.image_projection.data.copy_(ip_st["image_projection"].to(device=dev, dtype=core_e.image_projection.dtype))
    core_e.image_proj_norm.load_state_dict(ip_st["image_proj_norm"])

    vcfg = cfg["variant_cfg"]
    h_mod = GeoFiLMOrthoUnified(
        core_e.image_projection.shape[1], specs, cfg["geo_kwargs"],
        use_bridge=vcfg["use_bridge"],
        use_gram_schmidt=vcfg.get("use_gram_schmidt", True),
        use_query_router=vcfg.get("use_query_router", True),
        use_corn_size=vcfg.get("use_corn_size", True),
        use_patch_film=vcfg.get("use_patch_film", True),
    )
    h_mod.load_state_dict(torch.load(ckpt / HEAD_FILE, map_location="cpu"))
    h_mod.to(device=dev, dtype=torch.float16).eval()

    cache = {"query_embeds": None, "query_mask": None, "canon_logits": None, "ortho_aux": None}
    orig_encode = core_e._encode_image
    def wrapped_encode(pixel_values):
        out = orig_encode(pixel_values)
        bridged, _, lg = h_mod.forward_bridge(out, query_embeds=cache.get("query_embeds"),
                                              query_mask=cache.get("query_mask"))
        cache["canon_logits"] = lg
        cache["ortho_aux"] = h_mod.latest_aux
        return bridged.to(out.dtype)
    core_e._encode_image = wrapped_encode
    m._feat_cache = cache
    print(f"✓ {run_dir.name} | variant {cfg['variant']} | {len(specs)} kategori kanonik | gerbang:",
          h_mod.gate_mean_abs())
    return m, proc, h_mod, core_e, specs

eval_model, eval_proc, eval_head, eval_core, EVAL_SPECS = load_ortho_for_eval(TARGET_RUN)
EVAL_LABELER = CanonLabeler(EVAL_SPECS)

PRED_PATH  = EVAL_DIR / "predictions.json"
RAW_PATH   = EVAL_DIR / "predictions_raw.json"          # {idx: prosa} — format identik M1-M4 & B1
CANON_PATH = EVAL_DIR / "canon_preds.json"          # {idx: {kategori: [atom]}}
DIAG_PATH  = EVAL_DIR / "ortho_diag.json"           # {idx: {router_active, corn_tier, cos_glob, cos_spat}}
predictions = {int(k): v for k, v in json.loads(PRED_PATH.read_text()).items()} if PRED_PATH.exists() else {}
predictions_raw = {int(k): v for k, v in json.loads(RAW_PATH.read_text()).items()} if RAW_PATH.exists() else dict(predictions)
canon_preds = {int(k): v for k, v in json.loads(CANON_PATH.read_text()).items()} if CANON_PATH.exists() else {}
ortho_diags = {int(k): v for k, v in json.loads(DIAG_PATH.read_text()).items()} if DIAG_PATH.exists() else {}
todo = [i for i in range(len(te)) if i not in predictions]
print(f"Sudah ada {len(predictions)} | sisa {len(todo)}")

def resolve_image(img_id):
    stem = strip_jpg(img_id)
    cand = IMG_DIR / f"{stem}.jpg"
    if cand.exists(): return cand
    hit = list(IMG_DIR.glob(f"{stem}.*"))
    return hit[0] if hit else None

@torch.no_grad()
def run_generate_batch(prompts_list, imgs_list):
    inputs = eval_proc(text=prompts_list, images=imgs_list, return_tensors="pt", padding=True)
    inputs = {k: v.to(dev) for k, v in inputs.items()}
    inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)
    eval_model._feat_cache["query_embeds"] = eval_core.get_input_embeddings()(inputs["input_ids"]).detach()
    eval_model._feat_cache["query_mask"] = inputs.get("attention_mask")
    eval_model._feat_cache["canon_logits"] = None
    eval_model._feat_cache["ortho_aux"] = None
    # JANGAN kirim attention_mask ke .generate(): Florence-2 menyusun mask 577+L sendiri
    gen_ids = eval_model.generate(input_ids=inputs["input_ids"], pixel_values=inputs["pixel_values"], **GEN_CFG)
    txt = [a.strip() for a in eval_proc.batch_decode(gen_ids, skip_special_tokens=True)]
    lg = eval_model._feat_cache.get("canon_logits")
    ox = eval_model._feat_cache.get("ortho_aux")
    cp = canonical_predict(lg, EVAL_LABELER) if lg is not None else [dict() for _ in txt]
    dg = []
    if ox is not None:
        rp_np = (ox["router_probs"].detach().cpu().numpy() >= 0.5)
        cc_np = (ox["corn_p_cum"].detach().cpu().numpy() >= 0.5).sum(axis=-1)
        cg = float(ox["cos_glob_after"].detach().cpu().item())
        cs = float(ox["cos_spat_after"].detach().cpu().item())
        for b_i in range(len(txt)):
            act_cats = [EVAL_LABELER.cats[ci] for ci in range(EVAL_LABELER.C) if rp_np[b_i, ci]]
            rk = int(min(cc_np[b_i], len(POLYP_SIZE_ORDERED_TIERS) - 1))
            dg.append({"router_cats": act_cats, "corn_size_tier": POLYP_SIZE_ORDERED_TIERS[rk],
                       "cos_glob": cg, "cos_spat": cs})
    else:
        dg = [{} for _ in txt]
    return txt, cp, dg

def _flush():
    PRED_PATH.write_text(json.dumps({str(k): predictions[k] for k in sorted(predictions)}, indent=2))
    CANON_PATH.write_text(json.dumps({str(k): canon_preds[k] for k in sorted(canon_preds)}))
    DIAG_PATH.write_text(json.dumps({str(k): ortho_diags[k] for k in sorted(ortho_diags)}))

def _to_names(cp):
    inv = {c: EVAL_SPECS[c]["atoms"] for c in EVAL_LABELER.cats}
    return {c: sorted(inv[c][j] for j in js) for c, js in cp.items()}

BATCH_INF = 16 if (torch.cuda.is_available() and torch.cuda.get_device_properties(0).total_memory / 1e9 >= 30.0) else 8
for s in tqdm(range(0, len(todo), BATCH_INF), desc=f"Infer {TARGET_RUN.name[:28]}"):
    idxs = todo[s:s + BATCH_INF]
    imgs, prompts, valid = [], [], []
    for idx in idxs:
        row = te.iloc[idx]
        p_img = resolve_image(row["img_id"])
        if p_img is None:
            predictions[idx] = ""; continue
        imgs.append(Image.open(p_img).convert("RGB"))
        prompts.append(PROMPT_PREFIX + str(row["question"]).strip())
        valid.append(idx)
    if not valid:
        continue
    try:
        txt, cp, dg = run_generate_batch(prompts, imgs)
    except RuntimeError:
        torch.cuda.empty_cache()
        txt, cp, dg = [], [], []
        for p_txt, im in zip(prompts, imgs):
            t1, c1, d1 = run_generate_batch([p_txt], [im]); txt += t1; cp += c1; dg += d1
    for idx, t, c, d in zip(valid, txt, cp, dg):
        predictions[idx] = t
        canon_preds[idx] = _to_names(c)
        ortho_diags[idx] = d
    if (s // BATCH_INF) % 25 == 0:
        _flush()
_flush()
n_empty = sum(1 for v in predictions.values() if not str(v).strip())
print(f"\n✓ Inferensi selesai: {len(predictions)} baris ({n_empty} kosong) -> {PRED_PATH}")
print(">>> SEKARANG BUKA 07_Ortho_Unified_Evaluation.ipynb UNTUK EVALUASI LENGKAP <<<")
assert len(predictions) == len(te)

✓ 20261116-000618_O2_Ortho_Unified_Full_r32_s42 | variant O2_Ortho_Unified_Full | 16 kategori kanonik | gerbang: (0.12335, 0.24475)
Sudah ada 0 | sisa 15955


Infer 20261116-000618_O2_Ortho_Uni:   0%|          | 0/998 [00:00<?, ?it/s]


✓ Inferensi selesai: 15955 baris (0 kosong) -> /content/drive/MyDrive/DOKTORAL/KvasirVQAX1/experiments/Paper3_Ortho_Unified_r32/20261116-000618_O2_Ortho_Unified_Full_r32_s42/eval/predictions.json
>>> SEKARANG BUKA 07_Ortho_Unified_Evaluation.ipynb UNTUK EVALUASI LENGKAP <<<
